In [8]:
from langchain_huggingface import HuggingFaceEmbeddings,ChatHuggingFace,HuggingFaceEndpoint
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [10]:
@tool
def multiply(a:int,b:int)->int:
    """Tool will return the multiplication of two number a and b"""
    return a*b

In [11]:
print(multiply.invoke({"a":3,"b":5}))

15


In [12]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Tool will return the multiplication of two number a and b
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [13]:
from dotenv import load_dotenv
load_dotenv()

True

In [14]:
# Tool Binding
llm=HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-120b",
    task="text-generation"
)

model=ChatHuggingFace(llm=llm)

In [16]:
llm_with_tools=model.bind_tools([multiply])

In [20]:
llm_with_tools.invoke('Hi how are you')

AIMessage(content='Hello! I’m doing great, thank you. How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 132, 'total_tokens': 184}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_fa36a6363905c2b568a1', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dc706-adf2-7b83-bd4b-75be1b0e81cd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 132, 'output_tokens': 52, 'total_tokens': 184})

In [23]:
result=llm_with_tools.invoke('can you multiply 7 with 6')

In [24]:
result.tool_calls[0]

{'name': 'multiply',
 'args': {'a': 7, 'b': 6},
 'id': '29ea124b4',
 'type': 'tool_call'}

In [25]:
result.tool_calls[0]['args']

{'a': 7, 'b': 6}

In [26]:
multiply.invoke(result.tool_calls[0]['args'])

42

In [27]:
multiply.invoke({'name': 'multiply',
 'args': {'a': 7, 'b': 6},
 'id': '29ea124b4',
 'type': 'tool_call'})

ToolMessage(content='42', name='multiply', tool_call_id='29ea124b4')

Maintaining a history of conversation and using our tool output as input in our llm model

In [43]:
query=HumanMessage('can you multiply 7 with 6')

In [44]:
messages=[query]
messages

[HumanMessage(content='can you multiply 7 with 6', additional_kwargs={}, response_metadata={})]

In [45]:
result1=llm_with_tools.invoke(messages)

In [46]:
result1

AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":7,"b":6}', 'name': 'multiply', 'description': None}, 'id': '113a26809', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 136, 'total_tokens': 179}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_fa36a6363905c2b568a1', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dc716-d458-7621-9448-9a145a8372a5-0', tool_calls=[{'name': 'multiply', 'args': {'a': 7, 'b': 6}, 'id': '113a26809', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 136, 'output_tokens': 43, 'total_tokens': 179})

In [47]:
messages.append(result1)

In [48]:
messages

[HumanMessage(content='can you multiply 7 with 6', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":7,"b":6}', 'name': 'multiply', 'description': None}, 'id': '113a26809', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 136, 'total_tokens': 179}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_fa36a6363905c2b568a1', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dc716-d458-7621-9448-9a145a8372a5-0', tool_calls=[{'name': 'multiply', 'args': {'a': 7, 'b': 6}, 'id': '113a26809', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 136, 'output_tokens': 43, 'total_tokens': 179})]

In [49]:
tool_result=multiply.invoke(result1.tool_calls[0])

In [50]:
tool_result

ToolMessage(content='42', name='multiply', tool_call_id='113a26809')

In [51]:
messages.append(tool_result)

In [52]:
messages

[HumanMessage(content='can you multiply 7 with 6', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":7,"b":6}', 'name': 'multiply', 'description': None}, 'id': '113a26809', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 136, 'total_tokens': 179}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_fa36a6363905c2b568a1', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dc716-d458-7621-9448-9a145a8372a5-0', tool_calls=[{'name': 'multiply', 'args': {'a': 7, 'b': 6}, 'id': '113a26809', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 136, 'output_tokens': 43, 'total_tokens': 179}),
 ToolMessage(content='42', name='multiply', tool_call_id='113a26809')]

In [54]:
llm_with_tools.invoke(messages).content

'Sure! The result of multiplying 7 by 6 is **42**.'